In [1]:
import os
import pickle
import numpy as np

current_dir = os.getcwd()
nominal_parts_path = os.path.join(current_dir, 'GCAD', 'parts_nominal_backup.pkl')

# Load the nominal parameters (Our Prior Mean)
with open(nominal_parts_path, 'rb') as f:
    nominal_parts = pickle.load(f)

print(f"✅ Loaded nominal parts dictionary containing {len(nominal_parts)} parts.")

✅ Loaded nominal parts dictionary containing 24 parts.


In [2]:
import copy

def generate_targeted_perturbed_dictionary(nominal_dict, spread_factor=2.0):
    """
    Takes a nominal dictionary and returns a new dictionary where ONLY targeted
    parameters are scaled by a random log-uniform factor. All other parameters
    remain exactly at their nominal values.
    """
    # Create a deep copy so we don't accidentally modify the original nominal_dict
    perturbed_dict = copy.deepcopy(nominal_dict)
    
    # The specific parameters we are actively investigating
    targets = {'Z6': [0, 1], 'I13': [0]}
    
    for part_name, indices in targets.items():
        for idx in indices:
            # Generate a random scaling factor for THIS specific index
            log_spread = np.log10(spread_factor)
            random_exponent = np.random.uniform(-log_spread, log_spread)
            scaling_factor = 10 ** random_exponent
            
            # Apply the scaling factor
            perturbed_dict[part_name][idx] *= scaling_factor
            
    return perturbed_dict

# Test it
test_dict = generate_targeted_perturbed_dictionary(nominal_parts)
print("--- PERTURBATION TEST ---")
print(f"Nominal Z6: {nominal_parts['Z6']}")
print(f"Test Z6:    {test_dict['Z6']}")
print(f"Nominal I13: {nominal_parts['I13']}")
print(f"Test I13:    {test_dict['I13']}")

--- PERTURBATION TEST ---
Nominal Z6: [2.0e-02 5.8e+01 4.3e-02]
Test Z6:    [2.69361482e-02 9.85637341e+01 4.30000000e-02]
Nominal I13: [0.012]
Test I13:    [0.0080166]


In [3]:

# --- GENERATE THE ENSEMBLE ---
N_MODELS = 200 
SPREAD_FACTOR = 2.0 
belief_ensemble = []

print(f"\nGenerating {N_MODELS} targeted parameter dictionaries...")
np.random.seed(42) 
for i in range(N_MODELS):
    new_dict = generate_targeted_perturbed_dictionary(nominal_parts, spread_factor=SPREAD_FACTOR)
    belief_ensemble.append(new_dict)

print(f"✅ Created a Belief Ensemble of {len(belief_ensemble)} models.")


Generating 200 targeted parameter dictionaries...
✅ Created a Belief Ensemble of 200 models.


In [4]:
memory_dir = os.path.join(current_dir, 'active_learning_loop', 'al_memory')
if not os.path.exists(memory_dir):
    os.makedirs(memory_dir)

# Create a dynamic filename using our configuration variables
filename = f"prior_belief_cloud_N{N_MODELS}_spread{SPREAD_FACTOR}.pkl"
ensemble_path = os.path.join(memory_dir, filename)

with open(ensemble_path, 'wb') as f:
    pickle.dump(belief_ensemble, f)

print(f"✅ Saved prior belief ensemble to '{ensemble_path}'")

✅ Saved prior belief ensemble to 'c:\Users\KuangQi\Desktop\GCAD-SDL\GCAD-SDL\Self_Calibrating_Loop\active_learning_loop\al_memory\prior_belief_cloud_N200_spread2.0.pkl'


In [5]:
# --- SANITY CHECK: BOUNDARY AND DISTANCE CHECK ---
ground_truth_dir = os.path.join(current_dir, 'ground_truth')
with open(os.path.join(ground_truth_dir, 'true_parts.pkl'), 'rb') as f:
    true_parts = pickle.load(f)

print("\n--- BOUNDARY CHECK ---")
for part in ['Z6', 'I13']:
    for i in range(len(true_parts[part])):
        nominal = nominal_parts[part][i]
        truth = true_parts[part][i]
        lower_bound = nominal / SPREAD_FACTOR
        upper_bound = nominal * SPREAD_FACTOR
        is_inside = lower_bound <= truth <= upper_bound
        print(f"  {part}[{i}]: Truth={truth:.4f} | Cloud Bounds=[{lower_bound:.4f}, {upper_bound:.4f}] -> {'✅ INSIDE' if is_inside else '❌ OUTSIDE'}")

print("\n--- FINDING THE LUCKIEST GUESS IN GEN 0 ---")
min_dist = float('inf')
best_model_idx = -1

for idx, model in enumerate(belief_ensemble):
    # Calculate a simple relative error distance between this model and the Truth
    dist_z6 = np.sum(np.abs(model['Z6'] - true_parts['Z6']) / true_parts['Z6'])
    dist_i13 = np.sum(np.abs(model['I13'] - true_parts['I13']) / true_parts['I13'])
    total_dist = dist_z6 + dist_i13
    
    if total_dist < min_dist:
        min_dist = total_dist
        best_model_idx = idx

print(f"The closest model in Generation 0 is Model #{best_model_idx}.")
print(f"It has a combined relative parameter distance of {min_dist:.2f}")
print(f"  Model {best_model_idx} Z6: {belief_ensemble[best_model_idx]['Z6']}")
print(f"  Truth Z6:       {true_parts['Z6']}")
print(f"  Model {best_model_idx} I13: {belief_ensemble[best_model_idx]['I13']}")
print(f"  Truth I13:       {true_parts['I13']}")


--- BOUNDARY CHECK ---
  Z6[0]: Truth=0.0150 | Cloud Bounds=[0.0100, 0.0400] -> ✅ INSIDE
  Z6[1]: Truth=72.5000 | Cloud Bounds=[29.0000, 116.0000] -> ✅ INSIDE
  Z6[2]: Truth=0.0430 | Cloud Bounds=[0.0215, 0.0860] -> ✅ INSIDE
  I13[0]: Truth=0.0090 | Cloud Bounds=[0.0060, 0.0240] -> ✅ INSIDE

--- FINDING THE LUCKIEST GUESS IN GEN 0 ---
The closest model in Generation 0 is Model #78.
It has a combined relative parameter distance of 0.23
  Model 78 Z6: [1.31083489e-02 7.89509776e+01 4.30000000e-02]
  Truth Z6:       [1.50e-02 7.25e+01 4.30e-02]
  Model 78 I13: [0.00885509]
  Truth I13:       [0.009]
